# Finetune with Contrastive-Learned Text Encoder


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DIR"]        = "/kaggle/tmp"
os.environ["WANDB_CACHE_DIR"]  = "/kaggle/tmp"
os.environ["WANDB_CONFIG_DIR"] = "/kaggle/tmp"

In [ ]:
import os, sys

IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB  = False
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    pass

WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
print(f"Runtime: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Other'}")
print(f"Working dir: {WORK_DIR}")

os.system("pip install -q selfies wandb pyarrow")
os.system("pip install -q 'rdkit>=2023.3'")
os.system("pip install -q 'transformers>=4.40' 'sentence-transformers>=2.6'")
print("Dependencies ready.")

In [ ]:
import subprocess, shutil

GIT_BRANCH = "prompt-condition"
REPO_NAME  = "morpheus"
REPO_PATH  = os.path.join(WORK_DIR, REPO_NAME)

!git clone -b {GIT_BRANCH} https://github.com/vivaikmalik/morpheus.git {REPO_PATH}

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
os.chdir(REPO_PATH)

In [ ]:
import wandb
os.environ.pop("WANDB_MODE", None)

if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
elif IS_COLAB:
  from google.colab import userdata
  key = userdata.get('WANDB_API_KEY')
  wandb.login(key = key)
else:
  wandb.login()

In [ ]:
import zipfile

dataset_url = "https://huggingface.co/datasets/zjunlp/Mol-Instructions/resolve/main/data/Molecule-oriented_Instructions.zip"
zip_file    = "Molecule-oriented_Instructions.zip"
extract_dir = "./data_mol_instruct"

if not os.path.exists(extract_dir):
    print("Downloading dataset...")
    os.system(f"wget -q {dataset_url} -O {zip_file}")
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(extract_dir)
    print("Extraction complete.")
else:
    print("Dataset already present.")

In [ ]:
import pandas as pd
import json

json_path = os.path.join(extract_dir, "Molecule-oriented_Instructions",
                         "description_guided_molecule_design.json")
with open(json_path, 'r') as f:
    data = json.load(f)

df = pd.DataFrame(data)

def format_prompt(row):
    instruction = row.get('instruction', '')
    inp = row.get('input', '')
    return f"{instruction}\nInput: {inp}" if inp else instruction

df['prompt']   = df.apply(format_prompt, axis=1)
df['response'] = df['output']
df['split']    = df['metadata'].apply(lambda m: m['split'])

train_df_ = df[df['split'] == 'train'][['prompt', 'response']]
train_df  = train_df_.sample(frac=0.9, random_state=42)
val_df    = train_df_.drop(train_df.index)
test_df   = df[df['split'] == 'test'][['prompt', 'response']]

train_csv_path = os.path.join(WORK_DIR, "train.csv")
val_csv_path   = os.path.join(WORK_DIR, "val.csv")
train_df.to_csv(train_csv_path, index=False)
val_df.to_csv(val_csv_path,   index=False)

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

In [ ]:
DATASET_CSV = os.path.join(WORK_DIR, "train.csv")
assert os.path.exists(DATASET_CSV)
df_check = pd.read_csv(DATASET_CSV, nrows=3)
assert "prompt"   in df_check.columns
assert "response" in df_check.columns
print(f"Dataset OK — {len(pd.read_csv(DATASET_CSV)):,} rows")
print(df_check[["prompt", "response"]].to_string())

In [ ]:
from pathlib import Path

shutil.copy(train_csv_path, os.path.join(REPO_PATH, "train.csv"))
shutil.copy(val_csv_path,   os.path.join(REPO_PATH, "val.csv"))

checkpoint_dir     = os.path.join(REPO_PATH, "checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)
PRETRAINED_WEIGHTS = Path(checkpoint_dir) / "best_model.pt"
if PRETRAINED_WEIGHTS.exists():
    print(f"Pre-trained weights found: {PRETRAINED_WEIGHTS}")
else:
    print("WARNING: best_model.pt not found — will train from scratch.")

In [ ]:
from finetune.train_text_condition import train, CONFIG

CONFIG["data_path"]     = "train.csv"
CONFIG["val_data_path"] = "val.csv"
CONFIG["num_workers"]   = 8
CONFIG["batch_size"]    = 512
CONFIG["log_every"]     = 1
CONFIG["num_epochs"]    = 25
CONFIG["wandb_project"] = "morpheus-diffusion-finetune-contrastive"

# Set one of these to load the encoder
CONFIG["contrastive_artifact"]  = None  # Wnadb artifact name
CONFIG["contrastive_ckpt_path"] = None  # Local path

print("Final CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k:30s} = {v}")

print("\nStarting training...")
train()

In [ ]:
FINETUNED = os.path.join(REPO_PATH, "checkpoints", "best_finetuned_model.pt")

if os.path.exists(FINETUNED):
    dest = os.path.join(WORK_DIR, "best_finetuned_model_contrastive.pt")
    shutil.copy(FINETUNED, dest)
    print(f"Checkpoint saved to {dest}")
else:
    print("No checkpoint found — training may not have completed a validation step yet.")

In [ ]:
WANDB_ARTIFACT = None  # e.g. "marl-project/morpheus-diffusion-finetune-contrastive/finetuned-contrastive-valloss0.6500:v0"

_model_in_memory = "model" in dir() and model is not None  # noqa: F821

if not _model_in_memory:
    print("No model found in memory — attempting to load...")

    import torch, wandb, glob as _glob
    import selfies as sf
    from transformers import AutoTokenizer
    from tokenizer.chemicalTokenizer import ChemicalTokenizer
    from model.molecularDiffusionModel import MolecularDiffusionModel
    from finetune.train_text_condition import CONFIG

    device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer    = ChemicalTokenizer(os.path.join(REPO_PATH, "chemical_tokenizer.json"))
    hf_tokenizer = AutoTokenizer.from_pretrained(CONFIG["text_model"])
    max_length   = CONFIG["max_length"]

    LOCAL_CKPT = os.path.join(WORK_DIR, "best_finetuned_model_contrastive.pt")

    if os.path.exists(LOCAL_CKPT):
        ckpt_path = LOCAL_CKPT
        print(f"Found local checkpoint: {ckpt_path}")
    elif WANDB_ARTIFACT is not None:
        print(f"Downloading W&B artifact: {WANDB_ARTIFACT}")
        _run = wandb.init(project="morpheus-eval", job_type="eval", resume="allow")
        _art = _run.use_artifact(WANDB_ARTIFACT, type="model")
        _dir = _art.download(root=os.path.join(WORK_DIR, "wandb_artifact"))
        _pts = _glob.glob(os.path.join(_dir, "**", "*.pt"), recursive=True)
        assert _pts, f"No .pt file found in artifact at {_dir}"
        ckpt_path = sorted(_pts)[0]
        _run.finish()
        print(f"Artifact downloaded → {ckpt_path}")
    else:
        raise FileNotFoundError(
            f"No model in memory, no local checkpoint at {LOCAL_CKPT}, "
            "and WANDB_ARTIFACT is not set."
        )

    model = MolecularDiffusionModel(
        vocab_size      = CONFIG["vocab_size"],
        hidden_size     = CONFIG["hidden_size"],
        num_heads       = CONFIG["num_heads"],
        ffn_dim         = CONFIG["ffn_dim"],
        num_layers      = CONFIG["num_layers"],
        max_length      = CONFIG["max_length"],
        pad_token_id    = tokenizer.pad_token_id,
        text_model_name = CONFIG["text_model"],
        dropout         = 0.0,
    ).to(device)

    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"], strict=False)
    model.eval()
    print(f"Model loaded from: {ckpt_path}\n")

else:
    print("Model already in memory — skipping reload.")


In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "nltk", "python-Levenshtein"], check=True)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

import pandas as pd
import numpy as np
import torch
import selfies as sf
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit.Chem.Scaffolds import MurckoScaffold
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import Levenshtein as lev
from tqdm.auto import tqdm

TEST_CSV         = os.path.join(WORK_DIR, "test.csv")
EVAL_BATCH_SIZE  = 32
EVAL_CFG_SCALE   = 3.0
EVAL_NUM_STEPS   = 32
EVAL_TEMPERATURE = 1.0
MAX_ROWS         = None

assert os.path.exists(TEST_CSV), f"test.csv not found at {TEST_CSV}"
test_df = pd.read_csv(TEST_CSV)
if MAX_ROWS:
    test_df = test_df.head(MAX_ROWS)
print(f"Evaluating on {len(test_df):,} test rows")

def generate_selfies_batch(prompts: list) -> list:
    n = len(prompts)
    text_inputs = hf_tokenizer(
        prompts, padding=True, truncation=True, return_tensors="pt"
    ).to(device)
    text_padding_mask = (text_inputs["attention_mask"] == 0)
    with torch.no_grad():
        text_embeds = model.get_text_embeddings(
            text_inputs["input_ids"], text_inputs["attention_mask"]
        )
        null_embeds = model.null_token.expand(n, text_embeds.size(1), -1)
        input_ids   = torch.full((n, max_length), tokenizer.mask_token_id, device=device)
        t_vals      = torch.linspace(1.0, 0.0, EVAL_NUM_STEPS, device=device)
        for step_idx, t_val in enumerate(t_vals):
            step_t        = t_val.repeat(n).unsqueeze(-1)
            cond_logits   = model(input_ids, step_t, text_embeds,  text_padding_mask)
            uncond_logits = model(input_ids, step_t, null_embeds,  text_padding_mask)
            logits        = uncond_logits + EVAL_CFG_SCALE * (cond_logits - uncond_logits)
            if step_idx < EVAL_NUM_STEPS - 1:
                logits[:, :, tokenizer.mask_token_id] = float("-inf")
            probs      = torch.softmax(logits / EVAL_TEMPERATURE, dim=-1)
            sampled    = torch.distributions.Categorical(probs=probs).sample()
            confidence = torch.gather(probs, 2, sampled.unsqueeze(-1)).squeeze(-1)
            alpha_t     = (torch.cos(t_val * torch.pi / 2) ** 2).item()
            num_to_mask = int((1.0 - alpha_t) * max_length)
            if num_to_mask > 0 and step_idx < EVAL_NUM_STEPS - 1:
                _, mask_idx = torch.topk(confidence, num_to_mask, dim=-1, largest=False)
                sampled.scatter_(1, mask_idx, tokenizer.mask_token_id)
            input_ids = sampled
    out = []
    for i in range(n):
        ids = input_ids[i].cpu().tolist()
        if tokenizer.eos_token_id in ids:
            ids = ids[:ids.index(tokenizer.eos_token_id)]
        out.append(tokenizer.decode(ids))
    return out

all_prompts   = test_df["prompt"].tolist()
all_targets   = test_df["response"].tolist()
all_generated = []
for start in tqdm(range(0, len(all_prompts), EVAL_BATCH_SIZE), desc="Generating"):
    all_generated.extend(generate_selfies_batch(all_prompts[start:start + EVAL_BATCH_SIZE]))

smoother = SmoothingFunction().method1

def selfies_to_smiles(s):
    try:
        mol = Chem.MolFromSmiles(sf.decoder(s))
        return Chem.MolToSmiles(mol) if mol else None
    except Exception:
        return None

def mol_fp(smiles):
    if not smiles: return None
    mol = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048) if mol else None

def tanimoto(fp1, fp2):
    return DataStructs.TanimotoSimilarity(fp1, fp2) if (fp1 and fp2) else 0.0

def bleu_score(ref_tokens, hyp_tokens):
    return sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoother)

def token_retention_rate(s):
    try:
        orig = list(sf.split_selfies(s))
        if not orig: return 0.0
        mol = Chem.MolFromSmiles(sf.decoder(s))
        if mol is None: return 0.0
        rt = list(sf.split_selfies(sf.encoder(Chem.MolToSmiles(mol))))
        return sum(1 for t in orig if t in rt) / len(orig)
    except Exception:
        return 0.0

def scaffold_match(smi_gen, smi_ref):
    try:
        return (MurckoScaffold.MurckoScaffoldSmiles(smiles=smi_gen, includeChirality=False) ==
                MurckoScaffold.MurckoScaffoldSmiles(smiles=smi_ref, includeChirality=False))
    except Exception:
        return False

records = []
for gen_sf, ref_sf in zip(all_generated, all_targets):
    gen_smi = selfies_to_smiles(gen_sf)
    ref_smi = selfies_to_smiles(ref_sf)
    gen_fp  = mol_fp(gen_smi)
    ref_fp  = mol_fp(ref_smi)
    gen_tok = list(sf.split_selfies(gen_sf))
    ref_tok = list(sf.split_selfies(ref_sf))
    records.append({
        "generated_selfies" : gen_sf,
        "reference_selfies" : ref_sf,
        "generated_smiles"  : gen_smi or "",
        "reference_smiles"  : ref_smi or "",
        "valid"             : gen_smi is not None,
        "exact_match"       : (gen_sf.strip() == ref_sf.strip()) or (gen_smi is not None and gen_smi == ref_smi),
        "bleu"              : bleu_score(ref_tok, gen_tok),
        "levenshtein"       : lev.distance(gen_sf, ref_sf),
        "tanimoto"          : tanimoto(gen_fp, ref_fp),
        "token_retention"   : token_retention_rate(gen_sf),
        "scaffold_match"    : scaffold_match(gen_smi, ref_smi) if (gen_smi and ref_smi) else False,
        "norm_levenshtein"  : lev.distance(gen_sf, ref_sf) / max(len(gen_sf), len(ref_sf), 1),
    })

results_df = pd.DataFrame(records)
valid_mask = results_df["valid"]

metrics = {
    "Validity (%)"                            : valid_mask.mean() * 100,
    "Exact Match (%)"                         : results_df["exact_match"].mean() * 100,
    "Scaffold Match — valid pairs (%)"        : results_df.loc[valid_mask, "scaffold_match"].mean() * 100,
    "BLEU Score (avg)"                        : results_df["bleu"].mean(),
    "Levenshtein Distance (avg)"              : results_df["levenshtein"].mean(),
    "Normalised Levenshtein (avg)"            : results_df["norm_levenshtein"].mean(),
    "Tanimoto Similarity (avg, all)"          : results_df["tanimoto"].mean(),
    "Tanimoto Similarity (avg, valid pairs)"  : results_df.loc[valid_mask, "tanimoto"].mean()
                                                if valid_mask.any() else float("nan"),
    "Token Retention Rate (avg, generated)"   : results_df["token_retention"].mean(),
}

print()
print("=" * 60)
print("  MODEL EVALUATION  —  test.csv")
print("=" * 60)
for name, val in metrics.items():
    print(f"  {name:<45s}  {val:>8.4f}")
print("-" * 60)
print(f"  Total samples evaluated : {len(results_df):>8,}")
print(f"  Valid molecules         : {valid_mask.sum():>8,} / {len(results_df):,}")
print("=" * 60)

results_csv = os.path.join(WORK_DIR, "eval_results_contrastive.csv")
results_df.to_csv(results_csv, index=False)
print(f"\nDetailed per-sample results saved → {results_csv}")


# ChEBI-20 Evaluation

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# The dataset ground-truth is SMILES.  We keep the raw SMILES as "response"
# so that the evaluation cell can:
#   • convert GT SMILES → SELFIES  for token-retention-rate
#   • convert predicted SELFIES → SMILES  for every other metric
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, os, io, requests
import pandas as pd

subprocess.run(["pip", "install", "-q",
                "fcd_torch", "scikit-learn",
                "nltk", "python-Levenshtein"], check=True)

CHEBI20_BASE = (
    "https://raw.githubusercontent.com/blender-nlp/MolT5"
    "/main/ChEBI-20_data"
)

def _download_split(fname):
    url  = f"{CHEBI20_BASE}/{fname}"
    print(f"Downloading {fname} …")
    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    df = pd.read_csv(
        io.StringIO(resp.text), sep="\t", header=0,
        names=["cid", "smiles", "description"],
    ).dropna(subset=["smiles", "description"])
    return pd.DataFrame({
        "prompt"  : df["description"].values,
        "response": df["smiles"].values,   # ground-truth is SMILES
    })

chebi20_train_df = _download_split("train.txt")
chebi20_val_df   = _download_split("validation.txt")
chebi20_test_df  = _download_split("test.txt")

print(
    f"Train : {len(chebi20_train_df):,}  "
    f"Val   : {len(chebi20_val_df):,}  "
    f"Test  : {len(chebi20_test_df):,}"
)

chebi20_test_csv = os.path.join(WORK_DIR, "chebi20_test.csv")
chebi20_test_df.to_csv(chebi20_test_csv, index=False)
print(f"\nTest split saved → {chebi20_test_csv}")
print(chebi20_test_df.head(3).to_string())


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ChEBI-20 Evaluation
#
# Ground truth  : SMILES
# Model output  : SELFIES
#
# Token Retention Rate  → GT SMILES is first converted to SELFIES so we can
#                          compare at the SELFIES-token level.
# All other metrics     → predicted SELFIES is converted to canonical SMILES
#                          before comparison.
#
# Metrics
#   • SMILES BLEU (character-level, method1 smoothing)
#   • SMILES Levenshtein distance (raw + normalised)
#   • SMILES Exact Match rate
#   • Validity
#   • MACCS FTS   (MACCS-keys Tanimoto similarity)
#   • RDK FTS     (RDKit topological-fp Tanimoto similarity)
#   • Morgan FTS  (Morgan r=2, 2048-bit Tanimoto similarity)
#   • FCD         (Fréchet ChemNet Distance — lower is better)
#   • Text2Mol    (dual-encoder cosine similarity from MolT5 evaluation)
#   • Token Retention Rate
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, os, sys, warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import selfies as sf
import pandas as pd
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, MACCSkeys
from rdkit.Chem import RDKFingerprint
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import Levenshtein as lev
from tqdm.auto import tqdm

RDLogger.DisableLog("rdApp.*")

# Hyperparameters
CHEBI20_BATCH_SIZE  = 32
CHEBI20_CFG_SCALE   = 3.0
CHEBI20_NUM_STEPS   = 32
CHEBI20_TEMPERATURE = 1.0
CHEBI20_MAX_ROWS    = None 

eval_df = chebi20_test_df.copy()
if CHEBI20_MAX_ROWS:
    eval_df = eval_df.head(CHEBI20_MAX_ROWS)
print(f"ChEBI-20 test rows to evaluate: {len(eval_df):,}")

# Utility helpers
_smoother = SmoothingFunction().method1

def _canonical(smiles: str):
    """Return canonical SMILES or None if invalid."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        return Chem.MolToSmiles(mol) if mol else None
    except Exception:
        return None

def _selfies_to_smiles(selfies_str: str):
    """Decode SELFIES → canonical SMILES, return None on failure."""
    try:
        raw = sf.decoder(selfies_str)
        return _canonical(raw)
    except Exception:
        return None

def _smiles_to_selfies(smiles: str):
    """Encode canonical SMILES → SELFIES, return None on failure."""
    try:
        can = _canonical(smiles)
        return sf.encoder(can) if can else None
    except Exception:
        return None

def _maccs_fp(smiles):
    mol = Chem.MolFromSmiles(smiles) if smiles else None
    return MACCSkeys.GenMACCSKeys(mol) if mol else None

def _rdk_fp(smiles):
    mol = Chem.MolFromSmiles(smiles) if smiles else None
    return RDKFingerprint(mol) if mol else None

def _morgan_fp(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles) if smiles else None
    return AllChem.GetMorganFingerprintAsBitVect(mol, radius, n_bits) if mol else None

def _tanimoto(fp1, fp2) -> float:
    return DataStructs.TanimotoSimilarity(fp1, fp2) if (fp1 and fp2) else 0.0

def _smiles_bleu(ref: str, gen: str) -> float:
    """Character-level BLEU on SMILES strings."""
    return sentence_bleu(
        [list(ref)], list(gen) if gen else [],
        smoothing_function=_smoother,
    )

def _token_retention(ref_smiles: str, gen_selfies: str) -> float:
    """
    Fraction of GT SELFIES tokens (derived from GT SMILES) that are present
    in the generated SELFIES token multiset.
    """
    try:
        ref_sf  = _smiles_to_selfies(ref_smiles)
        if not ref_sf:
            return 0.0
        ref_tok = list(sf.split_selfies(ref_sf))
        gen_tok = list(sf.split_selfies(gen_selfies))
        if not ref_tok:
            return 0.0
        gen_tok_set = set(gen_tok)
        return sum(1 for t in ref_tok if t in gen_tok_set) / len(ref_tok)
    except Exception:
        return 0.0

# Generation
def _generate_batch(prompts: list) -> list:
    n = len(prompts)
    enc = hf_tokenizer(
        prompts, padding=True, truncation=True,
        max_length=512, return_tensors="pt",
    ).to(device)
    pad_mask = (enc["attention_mask"] == 0)

    with torch.no_grad():
        txt_emb  = model.get_text_embeddings(enc["input_ids"], enc["attention_mask"])
        null_emb = model.null_token.expand(n, txt_emb.size(1), -1)
        ids      = torch.full((n, max_length), tokenizer.mask_token_id, device=device)

        for step, t in enumerate(torch.linspace(1.0, 0.0, CHEBI20_NUM_STEPS, device=device)):
            st          = t.repeat(n).unsqueeze(-1)
            c_logits    = model(ids, st, txt_emb,  pad_mask)
            u_logits    = model(ids, st, null_emb, pad_mask)
            logits      = u_logits + CHEBI20_CFG_SCALE * (c_logits - u_logits)
            if step < CHEBI20_NUM_STEPS - 1:
                logits[:, :, tokenizer.mask_token_id] = float("-inf")
            probs  = torch.softmax(logits / CHEBI20_TEMPERATURE, dim=-1)
            samp   = torch.distributions.Categorical(probs=probs).sample()
            conf   = torch.gather(probs, 2, samp.unsqueeze(-1)).squeeze(-1)
            alpha  = (torch.cos(t * torch.pi / 2) ** 2).item()
            n_mask = int((1.0 - alpha) * max_length)
            if n_mask > 0 and step < CHEBI20_NUM_STEPS - 1:
                _, mi = torch.topk(conf, n_mask, dim=-1, largest=False)
                samp.scatter_(1, mi, tokenizer.mask_token_id)
            ids = samp

    out = []
    for i in range(n):
        tok = ids[i].cpu().tolist()
        if tokenizer.eos_token_id in tok:
            tok = tok[:tok.index(tokenizer.eos_token_id)]
        out.append(tokenizer.decode(tok))
    return out

prompts_list    = eval_df["prompt"].tolist()
ref_smiles_list = eval_df["response"].tolist()   # GT is SMILES
gen_selfies_list = []

for start in tqdm(range(0, len(prompts_list), CHEBI20_BATCH_SIZE),
                  desc="Generating (ChEBI-20)"):
    gen_selfies_list.extend(
        _generate_batch(prompts_list[start : start + CHEBI20_BATCH_SIZE])
    )

# Per-sample metrics
records = []
for ref_smi, gen_sf_str in zip(ref_smiles_list, gen_selfies_list):
    ref_can  = _canonical(ref_smi) or ref_smi   # canonical GT SMILES
    gen_smi  = _selfies_to_smiles(gen_sf_str)    # predicted SMILES

    valid    = gen_smi is not None
    exact    = valid and (gen_smi == ref_can)

    records.append({
        # raw strings
        "reference_smiles"        : ref_can,
        "generated_selfies"       : gen_sf_str,
        "generated_smiles"        : gen_smi or "",
        # binary flags
        "valid"                   : valid,
        "smiles_exact_match"      : exact,
        # SMILES-space string metrics
        "smiles_bleu"             : _smiles_bleu(ref_can, gen_smi) if valid else 0.0,
        "smiles_levenshtein"      : lev.distance(gen_smi or "", ref_can),
        "smiles_norm_levenshtein" : lev.distance(gen_smi or "", ref_can)
                                    / max(len(gen_smi or ""), len(ref_can), 1),
        # fingerprint Tanimoto similarities (0.0 when either molecule is invalid)
        "maccs_fts"               : _tanimoto(_maccs_fp(ref_can),  _maccs_fp(gen_smi)),
        "rdk_fts"                 : _tanimoto(_rdk_fp(ref_can),    _rdk_fp(gen_smi)),
        "morgan_fts"              : _tanimoto(_morgan_fp(ref_can), _morgan_fp(gen_smi)),
        # SELFIES-space token metric — GT SMILES converted to SELFIES internally
        "token_retention_rate"    : _token_retention(ref_can, gen_sf_str),
    })

results_df = pd.DataFrame(records)
valid_mask  = results_df["valid"]

# FCD
fcd_value = float("nan")
try:
    from fcd_torch import FCD as _FCD

    def _fcd_safe(smiles: str) -> bool:
        """Accept only SMILES that fcd_torch can encode (len >= 2, valid mol)."""
        return bool(smiles) and len(smiles) >= 2 and Chem.MolFromSmiles(smiles) is not None

    pairs = [
        (r["generated_smiles"], r["reference_smiles"])
        for r in records
        if r["valid"] and _fcd_safe(r["generated_smiles"]) and _fcd_safe(r["reference_smiles"])
    ]
    if len(pairs) >= 2:
        gen_valid, ref_valid = zip(*pairs)
        _fcd_scorer = _FCD(device=str(device), n_jobs=0)   # n_jobs=0 → main process
        fcd_value   = _fcd_scorer(list(gen_valid), list(ref_valid))
        print(f"FCD computed over {len(pairs):,} valid pairs → {fcd_value:.4f}")
    else:
        print(f"Too few FCD-compatible molecules ({len(pairs)}) — skipped.")
except ImportError:
    print("fcd_torch not installed — FCD skipped.")
except Exception as _e:
    print(f"FCD failed: {_e}")

# Text2Mol (MolT5 evaluation utilities)
text2mol_mean = float("nan")
try:
    _T2M_DIR = os.path.join(WORK_DIR, "molt5_eval")
    _SCORE_PY = os.path.join(_T2M_DIR, "text2mol_score.py")

    if not os.path.exists(_SCORE_PY):
        print("Cloning MolT5 evaluation utilities …")
        subprocess.run([
            "git", "clone", "--depth", "1", "--filter=blob:none",
            "--sparse", "https://github.com/blender-nlp/MolT5.git",
            _T2M_DIR,
        ], check=True, capture_output=True)
        subprocess.run(
            ["git", "sparse-checkout", "set", "evaluation"],
            cwd=_T2M_DIR, check=True, capture_output=True,
        )
        import shutil as _sh
        for _f in os.listdir(os.path.join(_T2M_DIR, "evaluation")):
            _sh.copy2(
                os.path.join(_T2M_DIR, "evaluation", _f), _T2M_DIR
            )

    if _T2M_DIR not in sys.path:
        sys.path.insert(0, _T2M_DIR)

    from text2mol_score import get_text2mol_score   # MolT5 evaluation helper

    _valid_triples = [
        (p, r["generated_smiles"], r["reference_smiles"])
        for p, r in zip(prompts_list, records)
        if r["valid"]
    ]
    if _valid_triples:
        _texts, _gen_s, _ref_s = zip(*_valid_triples)
        text2mol_mean = get_text2mol_score(
            list(_gen_s), list(_texts),
            ckpt_dir=_T2M_DIR,
            device=str(device),
        )
        print(f"Text2Mol score (mean cosine) → {text2mol_mean:.4f}")
    else:
        print("No valid generated molecules — Text2Mol skipped.")

except Exception as _e:
    print(f"Text2Mol metric skipped: {_e}")

# Aggregate & print
def _fmt(v):
    return f"{v:>9.4f}" if not (isinstance(v, float) and np.isnan(v)) else "      N/A"

chebi20_metrics = {
    "Validity (%)"                     : valid_mask.mean() * 100,
    "SMILES Exact Match (%)"           : results_df["smiles_exact_match"].mean() * 100,
    "SMILES BLEU (avg, valid only)"    : (results_df.loc[valid_mask, "smiles_bleu"].mean()
                                          if valid_mask.any() else float("nan")),
    "SMILES Levenshtein (avg)"         : results_df["smiles_levenshtein"].mean(),
    "SMILES Norm. Levenshtein (avg)"   : results_df["smiles_norm_levenshtein"].mean(),
    "MACCS FTS (avg)"                  : results_df["maccs_fts"].mean(),
    "RDK FTS   (avg)"                  : results_df["rdk_fts"].mean(),
    "Morgan FTS (avg)"                 : results_df["morgan_fts"].mean(),
    "Token Retention Rate (avg)"       : results_df["token_retention_rate"].mean(),
    "FCD  ↓ (lower is better)"         : fcd_value,
    "Text2Mol Score (avg cosine)"      : text2mol_mean,
}

print()
print("=" * 68)
print("  ChEBI-20 EVALUATION  —  80/10/10  test split")
print("=" * 68)
for name, val in chebi20_metrics.items():
    print(f"  {name:<42s}  {_fmt(val)}")
print("-" * 68)
print(f"  Total samples   : {len(results_df):>8,}")
print(f"  Valid molecules : {valid_mask.sum():>8,} / {len(results_df):,}")
print("=" * 68)

_out_csv = os.path.join(WORK_DIR, "chebi20_eval_results.csv")
results_df.to_csv(_out_csv, index=False)
print(f"\nPer-sample results → {_out_csv}")


# Interactive Inference

In [ ]:
import torch
import selfies as sf
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display
from transformers import AutoTokenizer

from tokenizer.chemicalTokenizer import ChemicalTokenizer
from model.molecularDiffusionModel import MolecularDiffusionModel
from finetune.train_text_condition import CONFIG

PROMPTS = [
    "A highly soluble molecule with a benzene ring.",
    "A small drug-like fragment with low molecular weight.",
    "A molecule containing a fluorine atom.",
    "A complex polycyclic aromatic compound.",
]

CFG_SCALE   = 3.0
NUM_STEPS   = 32
TEMPERATURE = 1.0

INFERENCE_CHECKPOINT = os.path.join(WORK_DIR, "best_finetuned_model_contrastive.pt")

device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer    = ChemicalTokenizer(os.path.join(REPO_PATH, "chemical_tokenizer.json"))
hf_tokenizer = AutoTokenizer.from_pretrained(CONFIG["text_model"])

model = MolecularDiffusionModel(
    vocab_size      = CONFIG["vocab_size"],
    hidden_size     = CONFIG["hidden_size"],
    num_heads       = CONFIG["num_heads"],
    ffn_dim         = CONFIG["ffn_dim"],
    num_layers      = CONFIG["num_layers"],
    max_length      = CONFIG["max_length"],
    pad_token_id    = tokenizer.pad_token_id,
    text_model_name = CONFIG["text_model"],
    dropout         = 0.0,
).to(device)

assert os.path.exists(INFERENCE_CHECKPOINT), \
    f"Checkpoint not found: {INFERENCE_CHECKPOINT}"
ckpt = torch.load(INFERENCE_CHECKPOINT, map_location=device)
model.load_state_dict(ckpt["model"], strict=False)
model.eval()
print(f"Loaded: {INFERENCE_CHECKPOINT}\n")

max_length = CONFIG["max_length"]
num_mols   = len(PROMPTS)

text_inputs       = hf_tokenizer(PROMPTS, padding=True, truncation=True, return_tensors="pt").to(device)
text_padding_mask = (text_inputs["attention_mask"] == 0)

with torch.no_grad():
    text_embeds = model.get_text_embeddings(text_inputs["input_ids"], text_inputs["attention_mask"])
    null_embeds = model.null_token.expand(num_mols, text_embeds.size(1), -1)
    input_ids   = torch.full((num_mols, max_length), tokenizer.mask_token_id, device=device)
    t_vals      = torch.linspace(1.0, 0.0, NUM_STEPS, device=device)

    for step_idx, t_val in enumerate(t_vals):
        step_t        = t_val.repeat(num_mols).unsqueeze(-1)
        cond_logits   = model(input_ids, step_t, text_embeds,  text_padding_mask)
        uncond_logits = model(input_ids, step_t, null_embeds,  text_padding_mask)
        logits        = uncond_logits + CFG_SCALE * (cond_logits - uncond_logits)
        if step_idx < NUM_STEPS - 1:
            logits[:, :, tokenizer.mask_token_id] = float('-inf')
        probs   = torch.softmax(logits / TEMPERATURE, dim=-1)
        sampled = torch.distributions.Categorical(probs=probs).sample()
        conf    = torch.gather(probs, 2, sampled.unsqueeze(-1)).squeeze(-1)
        alpha_t = (torch.cos(t_val * torch.pi / 2) ** 2).item()
        n_mask  = int((1.0 - alpha_t) * max_length)
        if n_mask > 0 and step_idx < NUM_STEPS - 1:
            _, mask_idx = torch.topk(conf, n_mask, dim=-1, largest=False)
            sampled.scatter_(1, mask_idx, tokenizer.mask_token_id)
        input_ids = sampled

mols = []
for i in range(num_mols):
    ids = input_ids[i].cpu().tolist()
    if tokenizer.eos_token_id in ids:
        ids = ids[:ids.index(tokenizer.eos_token_id)]
    selfies_str = tokenizer.decode(ids)
    try:
        smiles = sf.decoder(selfies_str)
        mol    = Chem.MolFromSmiles(smiles)
        if mol:
            mols.append(mol)
            print(f"[{i}] Valid   {Chem.MolToSmiles(mol)}")
        else:
            mols.append(None)
            print(f"[{i}] Invalid SELFIES: {selfies_str}")
    except Exception as e:
        mols.append(None)
        print(f"[{i}] Error: {e}")
    print(f"     Prompt: {PROMPTS[i]}")

valid_mols = [m for m in mols if m is not None]
if valid_mols:
    display(Draw.MolsToGridImage(valid_mols, molsPerRow=2, subImgSize=(300, 300)))